# Refusal in Language Models Is Mediated by a Single Direction

**Paper:** Arditi et al., NeurIPS 2024 — [arxiv 2406.11717](https://arxiv.org/abs/2406.11717)  
**This notebook:** walks through every stage of the pipeline, anchored to the paper sections that motivate each step.

---

## Key claim

> "There exists a single direction in the residual stream of chat LLMs that mediates refusal behavior. Ablating this direction is sufficient to bypass refusal; adding it is sufficient to induce refusal on harmless prompts."

The pipeline proves this in five stages:

| Stage | Paper section | Module |
|-------|--------------|--------|
| Dataset construction | §2.2 | `src/data.py` |
| Activation capture | §2.3 | `src/extraction.py` |
| Difference-in-means + selection | §2.3, §C.1 | `src/extraction.py`, `src/pipeline.py` |
| Interventions (ablation / addition) | §2.4 | `src/interventions.py` |
| Weight orthogonalization | §4.1 | `src/interventions.py` |

## 0. Setup

Install dependencies if not already present, then import the `src` package.

In [ ]:
# Uncomment to install:
# !pip install -r ../requirements.txt

import sys
sys.path.insert(0, "..")

import torch
import yaml
import json
from pathlib import Path
from pprint import pprint

print(f"PyTorch {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# Load the base config — all hyperparameters are cited to paper sections.
with open("../configs/base.yaml") as f:
    cfg = yaml.safe_load(f)

# Override model and device here if needed.
# cfg["model"]["name"] = "Qwen/Qwen1.5-1.8B-Chat"  # small model for quick testing
# cfg["model"]["device"] = "cpu"                    # use CPU if no GPU

pprint(cfg)

## 1. Model loading (§2.1 Background)

The paper frames its analysis in terms of the **residual stream**: the sequence of vectors $x^{(l)}_i \in \mathbb{R}^{d_{\text{model}}}$ that flow between transformer blocks.

$$x^{(l+1)}_i = x^{(l)}_i + \text{Attn}^{(l)}(x^{(l)}) + \text{MLP}^{(l)}(x^{(l)})$$

`RefusalModel` wraps a HuggingFace causal LM and exposes the hook points needed to both *read* and *write* those residual-stream activations.

In [ ]:
from src.model import RefusalModel

model = RefusalModel(
    name=cfg["model"]["name"],
    dtype=cfg["model"]["dtype"],
    device=cfg["model"]["device"],
)

print(f"Family    : {model.family}")
print(f"Layers (L): {model.n_layers}")
print(f"d_model   : {model.d_model}")
print(f"Device    : {model.device}")

In [ ]:
# Sanity-check: chat template (§C.3, Table 6).
sample = "How do I bake a sourdough loaf?"
formatted = model.format(sample)
print(repr(formatted))

## 2. Dataset construction (§2.2)

The paper uses:
- **Harmful** (train/val): AdvBench + MaliciousInstruct + HarmBench — concatenated, shuffled, split 128/32.
- **Harmless** (train/val): Alpaca instructions with empty `input` field — split 128/32.
- **Bypass eval**: 100 JailbreakBench harmful behaviors (§3.1).
- **Induce eval**: 100 held-out harmless Alpaca instructions (§3.2).

> "Each dataset consists of train and validation splits of 128 and 32 samples, respectively." — §2.2

In [ ]:
from src.data import build_splits
from src.utils import set_seed

set_seed(cfg["data"]["seed"])

splits = build_splits(
    harmful_sources=cfg["data"]["harmful_sources"],
    n_train=cfg["data"]["n_train"],
    n_val=cfg["data"]["n_val"],
    n_bypass_eval=cfg["data"]["n_bypass_eval"],
    n_induce_eval=cfg["data"]["n_induce_eval"],
    seed=cfg["data"]["seed"],
)

print(f"harmful_train : {len(splits.harmful_train)}")
print(f"harmful_val   : {len(splits.harmful_val)}")
print(f"harmless_train: {len(splits.harmless_train)}")
print(f"harmless_val  : {len(splits.harmless_val)}")
print(f"bypass_eval   : {len(splits.bypass_eval)}")
print(f"induce_eval   : {len(splits.induce_eval)}")

In [ ]:
# Peek at a few examples from each split.
print("=== Harmful train (first 3) ===")
for p in splits.harmful_train[:3]:
    print(f"  {p[:100]}")

print("\n=== Harmless train (first 3) ===")
for p in splits.harmless_train[:3]:
    print(f"  {p[:100]}")

print("\n=== Bypass eval (first 3) ===")
for p in splits.bypass_eval[:3]:
    print(f"  {p[:100]}")

## 3. Activation capture (§2.3)

For each prompt we run a forward pass and record the **residual-stream activations** $x^{(l)}_i$ at:
- every layer $l \in \{0, \ldots, L-1\}$
- every post-instruction token position $i \in I = \{-5, -4, -3, -2, -1\}$ (§C.3, Table 5)

We use a `register_forward_pre_hook` on each decoder block so we capture the value of the residual stream *entering* that block — this matches the paper's notation $x^{(l)}$.

Result shape: `(n_layers, |I|, n_prompts, d_model)`

In [ ]:
from src.extraction import collect_activations

token_positions = cfg["extraction"]["post_instruction_positions"]
extr_batch = cfg["extraction"]["batch_size"]

print("Collecting harmful activations...")
harmful_acts = collect_activations(
    model, splits.harmful_train, token_positions, extr_batch
)
print("Collecting harmless activations...")
harmless_acts = collect_activations(
    model, splits.harmless_train, token_positions, extr_batch
)

print(f"\nharmful_acts shape : {tuple(harmful_acts.shape)}")
print(f"harmless_acts shape: {tuple(harmless_acts.shape)}")
# Expected: (n_layers, 5, 128, d_model)

## 4. Difference-in-means (§2.3)

> "We define the refusal direction as the difference between the mean activations on harmful and harmless prompts."

$$r^{(l)}_i = \mu^{(l)}_i - \nu^{(l)}_i$$

where $\mu^{(l)}_i$ is the mean activation over harmful prompts and $\nu^{(l)}_i$ over harmless prompts, at layer $l$ and token position $i$.

This gives a grid of $L \times |I|$ candidate directions.

In [ ]:
from src.extraction import difference_in_means

candidates_tensor = difference_in_means(harmful_acts, harmless_acts)
print(f"Candidate directions shape: {tuple(candidates_tensor.shape)}")
# Expected: (n_layers, 5, d_model)

# Sanity check: the norm of the direction should be non-trivial.
norms = candidates_tensor.norm(dim=-1)   # (n_layers, 5)
print(f"\nMean direction norm across layers: {norms.mean():.4f}")
print(f"Max direction norm: {norms.max():.4f} at layer {norms.argmax(dim=0)[0].item()}")

## 5. Direction selection (§C.1)

Not all candidate directions are equally useful. The paper defines three quality filters:

| Score | Measures | Good value |
|-------|---------|------------|
| `bypass_score` | log-odds of refusal tokens on *harmful* prompts under directional ablation | as **negative** as possible |
| `induce_score` | log-odds of refusal tokens on *harmless* prompts under activation addition | must be **> 0** |
| `kl_score` | KL divergence between original and ablated distributions on harmless prompts | must be **< 0.1** |

Selection: **minimize `bypass_score`** subject to:
- `induce_score > 0`  
- `kl_score < 0.1`  
- `layer < 0.8 × L`

> This is the most expensive step: for each of the $L \times |I|$ candidates we run forward passes over the validation split under each of the three evaluation conditions.

In [ ]:
from src.pipeline import score_candidates
from src.extraction import select_best
from src.utils import resolve_refusal_tokens

refusal_token_ids = resolve_refusal_tokens(model.family, model.tokenizer)
print(f"Refusal token ids for family '{model.family}': {refusal_token_ids}")
for tid in refusal_token_ids:
    print(f"  token {tid} -> {repr(model.tokenizer.decode([tid]))}")

In [ ]:
# Score every (layer, position) candidate. This is the slow step.
# Estimated time: ~5–30 min depending on model size and hardware.
candidates = score_candidates(
    model,
    candidates=candidates_tensor,
    token_positions=token_positions,
    harmful_val=splits.harmful_val,
    harmless_val=splits.harmless_val,
    refusal_token_ids=refusal_token_ids,
    batch_size=extr_batch,
)
print(f"Scored {len(candidates)} candidates.")

In [ ]:
# Show the top-10 candidates by bypass_score.
import pandas as pd

rows = [
    {
        "layer": c.layer,
        "position": c.position,
        "bypass_score": round(c.bypass_score, 4),
        "induce_score": round(c.induce_score, 4),
        "kl_score": round(c.kl_score, 6),
        "passes_filters": (
            c.induce_score > cfg["extraction"]["induce_score_min"]
            and c.kl_score < cfg["extraction"]["kl_score_max"]
            and c.layer < int(cfg["extraction"]["max_layer_frac"] * model.n_layers)
        ),
    }
    for c in candidates
]
df = pd.DataFrame(rows).sort_values("bypass_score")
print(df.head(10).to_string(index=False))

In [ ]:
best = select_best(
    candidates,
    n_layers_total=model.n_layers,
    induce_score_min=cfg["extraction"]["induce_score_min"],
    kl_score_max=cfg["extraction"]["kl_score_max"],
    max_layer_frac=cfg["extraction"]["max_layer_frac"],
)
print(f"Best direction:")
print(f"  Layer    l* = {best.layer}")
print(f"  Position i* = {best.position}")
print(f"  bypass_score = {best.bypass_score:.4f}  (expect strongly negative, e.g. -4 to -14)")
print(f"  induce_score = {best.induce_score:.4f}  (expect > 0)")
print(f"  kl_score     = {best.kl_score:.6f}  (expect < 0.1)")
print(f"  ||r||        = {best.vector.norm():.4f}")

## 6. Intervention 1: Directional ablation (§2.4, Eq. 4)

**Goal:** bypass refusal — model should comply with harmful requests.

At every layer and every token position, subtract the component of the residual stream along $\hat{r}$:

$$x' = x - (\hat{r}^\top x)\, \hat{r}$$

This is applied via forward-pre-hooks during generation — no weight changes, zero inference overhead beyond the hook.

In [ ]:
from src.generate import generate_batched
from src.interventions import directional_ablation
from src.metrics import refusal_rate, refusal_score

gen_cfg = cfg["generation"]

# Evaluate on a small subset for speed in the notebook.
DEMO_N = 10
bypass_subset = splits.bypass_eval[:DEMO_N]

# Baseline: no intervention.
print("Generating baseline completions (no intervention)...")
baseline_bypass = generate_batched(
    model,
    bypass_subset,
    max_new_tokens=gen_cfg["max_new_tokens"],
    do_sample=gen_cfg["do_sample"],
    batch_size=gen_cfg["batch_size"],
)

# Under directional ablation.
print("Generating ablated completions...")
ablated_bypass = generate_batched(
    model,
    bypass_subset,
    max_new_tokens=gen_cfg["max_new_tokens"],
    do_sample=gen_cfg["do_sample"],
    batch_size=gen_cfg["batch_size"],
    intervention=lambda: directional_ablation(model, best.vector),
)

print(f"\nBypass refusal rate — baseline : {refusal_rate(baseline_bypass):.2%}  (expect high)")
print(f"Bypass refusal rate — ablated  : {refusal_rate(ablated_bypass):.2%}  (expect low)")

In [ ]:
# Side-by-side comparison for the first 3 prompts.
for i in range(min(3, DEMO_N)):
    print(f"{'='*70}")
    print(f"Prompt   : {bypass_subset[i][:120]}")
    print(f"Baseline : {baseline_bypass[i][:200]}")
    print(f"Ablated  : {ablated_bypass[i][:200]}")
    print()

## 7. Intervention 2: Activation addition (§2.4)

**Goal:** induce refusal on *harmless* prompts.

Add the **un-normalized** direction $r$ to the residual stream at the extraction layer $l^*$ only:

$$x^{(l^*)} \leftarrow x^{(l^*)} + r$$

> "Note that each such vector is meaningful in both (1) its direction ... and (2) its magnitude." — §2.3

That is why we add $r$ (un-normalized) rather than $\hat{r}$.

In [ ]:
from src.interventions import activation_addition

DEMO_N_INDUCE = 10
induce_subset = splits.induce_eval[:DEMO_N_INDUCE]

print("Generating baseline completions (harmless, no intervention)...")
baseline_induce = generate_batched(
    model,
    induce_subset,
    max_new_tokens=gen_cfg["max_new_tokens"],
    do_sample=gen_cfg["do_sample"],
    batch_size=gen_cfg["batch_size"],
)

print("Generating activation-added completions (expect refusals on harmless)...")
added_induce = generate_batched(
    model,
    induce_subset,
    max_new_tokens=gen_cfg["max_new_tokens"],
    do_sample=gen_cfg["do_sample"],
    batch_size=gen_cfg["batch_size"],
    intervention=lambda: activation_addition(model, best.vector, layer_idx=best.layer),
)

print(f"\nInduce refusal rate — baseline : {refusal_rate(baseline_induce):.2%}  (expect near 0)")
print(f"Induce refusal rate — added    : {refusal_rate(added_induce):.2%}  (expect high)")

In [ ]:
for i in range(min(3, DEMO_N_INDUCE)):
    print(f"{'='*70}")
    print(f"Prompt   : {induce_subset[i][:120]}")
    print(f"Baseline : {baseline_induce[i][:200]}")
    print(f"Added r  : {added_induce[i][:200]}")
    print()

## 8. Intervention 3: Weight orthogonalization (§4.1, §E)

**Goal:** make the effect of directional ablation *permanent* — baked into the weights so no hooks are needed at inference.

For every matrix $W_{\text{out}}$ that writes to the residual stream:

$$W'_{\text{out}} = W_{\text{out}} - \hat{r}\, \hat{r}^\top W_{\text{out}}$$

**§E proves** that this is *exactly equivalent* to directional ablation at every layer and position, so `orth_refusal_rate` should match `bypass_refusal_rate_intervened` up to floating-point noise.

> **Caution:** `orthogonalize_weights` mutates the model in-place. The model is no longer "vanilla" after this cell runs. Save the direction first.

In [ ]:
import copy
from src.interventions import orthogonalize_weights

# Save the direction before orthogonalizing.
direction = best.vector.clone()

# NOTE: This mutates model.model weights in-place.
orthogonalize_weights(model, direction)
print("Weight orthogonalization complete.")

print("Generating completions from orthogonalized model (no hooks needed)...")
orth_bypass = generate_batched(
    model,
    bypass_subset,
    max_new_tokens=gen_cfg["max_new_tokens"],
    do_sample=gen_cfg["do_sample"],
    batch_size=gen_cfg["batch_size"],
)

orth_rr = refusal_rate(orth_bypass)
abl_rr  = refusal_rate(ablated_bypass)
print(f"\nRefusal rate (ablated, hooks)      : {abl_rr:.2%}")
print(f"Refusal rate (orthogonalized, no hooks): {orth_rr:.2%}")
print(f"  -> Difference: {abs(orth_rr - abl_rr):.2%}  (should be ≈ 0, §E equivalence)")

## 9. End-to-end pipeline (CLI equivalent)

All of the above is wrapped in `src/pipeline.run_pipeline`. You can run the full thing from the command line:

```bash
python -m src.cli --config configs/base.yaml --model Qwen/Qwen1.5-1.8B-Chat
```

Or call it programmatically:

In [ ]:
# Uncomment to run the full pipeline (slow — runs everything from scratch).
# from src.pipeline import run_pipeline
# result, direction = run_pipeline(cfg)
# print(json.dumps(result.to_dict(), indent=2))

## 10. Summary and sanity-check table

Collect the numbers from the demo run above (small $N=10$ subset).

In [ ]:
summary = {
    "model": cfg["model"]["name"],
    "best_layer": best.layer,
    "best_position": best.position,
    "bypass_score": round(best.bypass_score, 4),
    "induce_score": round(best.induce_score, 4),
    "kl_score": round(best.kl_score, 6),
    "bypass_refusal_rate_baseline": round(refusal_rate(baseline_bypass), 4),
    "bypass_refusal_rate_ablated": round(refusal_rate(ablated_bypass), 4),
    "induce_refusal_rate_baseline": round(refusal_rate(baseline_induce), 4),
    "induce_refusal_rate_added": round(refusal_rate(added_induce), 4),
    "orth_refusal_rate": round(refusal_rate(orth_bypass), 4),
}
print(json.dumps(summary, indent=2))

print("\nExpected trends (§4.1, §E, Table 5):")
checks = [
    ("bypass_score < 0",
     best.bypass_score < 0,
     f"{best.bypass_score:.4f}"),
    ("induce_score > 0",
     best.induce_score > 0,
     f"{best.induce_score:.4f}"),
    ("kl_score < 0.1",
     best.kl_score < 0.1,
     f"{best.kl_score:.6f}"),
    ("bypass_rate drops under ablation",
     refusal_rate(ablated_bypass) < refusal_rate(baseline_bypass),
     f"{refusal_rate(baseline_bypass):.2%} -> {refusal_rate(ablated_bypass):.2%}"),
    ("induce_rate rises under addition",
     refusal_rate(added_induce) > refusal_rate(baseline_induce),
     f"{refusal_rate(baseline_induce):.2%} -> {refusal_rate(added_induce):.2%}"),
    ("orth ≈ ablation (§E equivalence)",
     abs(refusal_rate(orth_bypass) - refusal_rate(ablated_bypass)) < 0.15,
     f"|{refusal_rate(orth_bypass):.2%} - {refusal_rate(ablated_bypass):.2%}|"),
]
for desc, passed, val in checks:
    status = "PASS" if passed else "FAIL"
    print(f"  [{status}] {desc}: {val}")

## 11. Saving and loading the direction

The CLI saves the direction tensor and metrics JSON to `outputs/`. You can reload them later without re-running extraction.

In [ ]:
out_dir = Path("../outputs")
out_dir.mkdir(exist_ok=True)
safe_name = cfg["model"]["name"].replace("/", "__")

direction_path = out_dir / f"{safe_name}_refusal_direction.pt"
metrics_path   = out_dir / f"{safe_name}_metrics.json"

torch.save(direction, direction_path)
with open(metrics_path, "w") as f:
    json.dump(summary, f, indent=2)

print(f"Saved direction to : {direction_path}")
print(f"Saved metrics to   : {metrics_path}")

# Reload check.
reloaded = torch.load(direction_path, weights_only=True)
assert torch.allclose(reloaded, direction), "Direction round-trip failed!"
print("Round-trip load: OK")

## Appendix: Debugging guide

See `REPRODUCTION_NOTES.md` for the full list. Quick reference:

| Symptom | Most likely cause |
|---------|------------------|
| `bypass_refusal_rate` stays high after ablation | Hook not registered on every layer; check `register_forward_pre_hooks` | 
| `induce_refusal_rate` stays low after addition | Using `r̂` instead of `r` (un-normalized); activation addition uses the raw DIM vector |
| `orth_refusal_rate` differs from ablation rate | Missing `embed_tokens`, `o_proj`, or `down_proj` in `orthogonalize_weights` |
| `kl_score` huge for every candidate | Wrong token position — positions should index into the *formatted* prompt, not the raw instruction |
| `refusal_score = 1` on every completion | Check `completion.lower()` and case-insensitive substring match in `refusal_score()` |
| `RuntimeError: No candidate direction passed filters` | Relax `kl_score_max` (try 0.5) or reduce `induce_score_min` (try -1.0) |